In [1]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import plot_utils as pu

In [2]:
# configure the backend for matplotlib
# this one should allow zoom:
# %matplotlib widget
# to make that work you need: "pip install ipympl" and run "jupyter nbextension enable --py widgetsnbextension"

# this will work without the above dependencies but won't allow zoom
%matplotlib inline

# this option may work whenever they fix bugs in mpld3
# import mpld3
# mpld3.enable_notebook()

In [51]:
# A function to create cuts given a target point
def add_cut(relaxed: gp.Model, integer_vars, integer_idx, plotter):
    
    tol = relaxed.params.FeasibilityTol
    all_done = True
    for iv in integer_vars:
        x = iv.X.item()
        if not np.isclose(x, round(x), atol=tol):
            all_done = False
            break
    if all_done:
        return False
    
    if plotter is not None:
        plotter.add_ball(1.2)
    variables = relaxed.getVars()  # TODO: pass this in as it's expensive?
    constraints = relaxed.getConstrs()  # wish we didn't have to use this one
    
    def find_variable_value(index):
        if index < len(variables):
            if variables[index].VBasis == -2:
                return 0.0  # not yet sure what to do here
            return variables[index].X
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        try:
            return constraint.Slack
        except:
            return 0.0
    
    def find_variable(index):
        if index < len(variables):
            # handle inverted variables (SCIP and Gurobi both have this silliness)
            if variables[index].VBasis == -2:  # not yet sure this is correct
                return variables[index].X - variables[index]
            assert variables[index].VBasis == -1  # not handling -3 yet
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        lhs, sense, rhs = relaxed.getRow(constraint), constraint.Sense, constraint.RHS
        if sense == '<':
             return rhs - lhs
        elif sense == '>':
            return lhs - rhs
        else:
            return 0.0  # is this right for equality constraints?
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=1)
    negated_vars = [basis[nr] for nr in negated_rows]
    
    # drop the rows of non-integer variables:
    # to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    # tableau = np.delete(tableau, to_drop, axis=0)
    # basis = np.delete(basis, to_drop)

    # Conforti has negative vectors with 1 at row=col, with the rest negated.
    if relaxed.Sense == gp.GRB.MINIMIZE:  # shouldn't have to check sense here; something ain't right
        tableau = -tableau
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = 1  # use our extra row to store the col==row -> 1
    
    col_to_var = [find_variable(c) for c in col_to_var]
    
    # left off:
    # we need to loop through all the integer variables and flag those with fractional portion
    # then for each of those we find a set of points on the rays.
    # we need/want to have at least as many points as we have rows in the basis. 
    # we run least squares to find the plane through all of them.
    # we then sort the cuts by distance from our original point and keep the p best as cuts.
    # we might need to verify that they aren't the same.
    # we need to ignore rays with a 0 component in the line we care about.
    # we also might not want to use least squares, but rather get just enough points.
    # review our notebook on living in the space of the basis. what does our cut look like there?
    # basis_corner = np.zeros(len(variables) + len(constraints))
    # for i in basis:
    #     basis_corner[i] = find_variable_value(i)
    to_cut = [(i, base, find_variable_value(base)) for i, base in enumerate(basis)  # can't call find_var_value after addConstr or update
              if base in integer_idx and tol < (integer_vars[integer_idx[base]].X % 1) < 1.0-tol]
    for row, base, x in to_cut:
        points = []
        for col, ray in enumerate(tableau.T):
            if np.isclose(ray[row], 0, atol=tol):
                continue
            scale = -(x % 1) / ray[row] if ray[row] < 0 else (1-(x % 1)) / ray[row]
            # print("SCALE", scale, x, r)
            # point = basis_corner.copy()
            # point[basis] += scale * ray[:-1]
            # point[r] += scale * ray[-1]
            points.append((col_to_var[col], 1.0/scale))
        # a = np.vstack(points)
        # y, _, _, _ = np.linalg.lstsq(a, np.zeros(a.shape[0]), rcond=tol)
        # left off: our old code probably handled distance down the ray; we need that. See if it was ball-dependant.
        # and also see if it drew the correct thing, and what space it drew in.
        summed_terms = gp.quicksum(nrm * v for v, nrm in points)
        new_con = relaxed.addConstr(summed_terms >= 1)

        # yd = np.linalg.det(a)
        # y = np.linalg.pinv(a, rcond=tol) @ np.ones(a.shape[0])
        # 
        # we assume order variables then slacks and that our point has all variables in it:
        
        # summed_terms = y[:len(variables)] @ variables + y[len(variables):-1] @ slack_variables
            
        if plotter is not None:
            relaxed.update()
            plotter.add_constraint(new_con)
            
    return True

def run_cuts_to_relaxed_sol(instances, loops=1):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.update()
        plotter = None #  pu.create(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)

        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break
                
            print("   Loop:", i+1, ", Score:", model.getObjective().getValue())
            if not add_cut(model, int_vars, int_idx, plotter):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break

        print("   Known best score:", instance.score if instance.known_optimum else "unknown")    
        if plotter is not None:
            plotter.render()

# test the cuts on simple examples:
run_cuts_to_relaxed_sol(list(el.get_instances().values())[4:5], loops=10)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom (upper bounded x)
   Relaxed 2 variables on 2D from bottom (upper bounded x)
   Loop: 1 , Score: 2.111111111111111
   Loop: 2 , Score: 1.9999999999999996
   Loop: 3 , Score: 1.1111111111111112
   Loop: 4 , Score: 0.7878787878787876
   Loop: 5 , Score: 0.5757575757575752

In [53]:
import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz3']]
run_cuts_to_relaxed_sol(jsplib_subset, loops=100)

Set parameter AggFill to value 10
Set parameter GomoryPasses to value 1
Running model: abz3
   Relaxed 27 variables on abz3
   Loop: 1 , Score: 250.0
   Loop: 2 , Score: 250.0
   Loop: 3 , Score: 280.92590894162345
   Loop: 4 , Score: 297.7258674667994
   Loop: 5 , Score: 305.56908398759305
   Loop: 6 , Score: 316.0714295640303
   Loop: 7 , Score: 316.6592025021744
   Loop: 8 , Score: 318.9447334945512
   Loop: 9 , Score: 320.7536993572802
   Loop: 10 , Score: 321.64358290595067
   Loop: 11 , Score: 323.00832296768766
   Loop: 12 , Score: 323.24881471935566
   Loop: 13 , Score: 323.59163669384617
   Loop: 14 , Score: 323.9272184371268
   Loop: 15 , Score: 324.57244786506647
   Loop: 16 , Score: 325.0178155100032
   Loop: 17 , Score: 326.1292979245946
   Loop: 18 , Score: 326.61798424785746
   Loop: 19 , Score: 326.8719106900599
   Loop: 20 , Score: 327.5059318941856
   Loop: 21 , Score: 327.93228329390126
   Loop: 22 , Score: 328.32124891066945
   Loop: 23 , Score: 328.4927355252735
  